In [ ]:
import csv
import pandas as pd
import os

os.chdir("C:/Users/yjshi/OneDrive/Tài liệu/Program/stock/data")
    
s1="115_1.csv"
s2="115_2.csv"
s3="115_3.csv"

# 讀入第一季資料
data1 = pd.read_csv(s1, encoding="utf-8")
data1 = data1.set_index("公司代號")

# 讀入第二季資料
data2 = pd.read_csv(s2, encoding="utf-8")
data2 = data2.set_index("公司代號")
    
data3 = pd.read_csv(s3, encoding="utf-8")
data3 = data3.set_index("公司代號")

growth_list = []

for code in data2.index:  # 以第二季為基準迭代
    if code in data1.index and code in data3.index:

        # 讀三季營收
        r1 = float(data1.loc[code, "營業收入-當月營收"])
        r2 = float(data2.loc[code, "營業收入-當月營收"])
        r3 = float(data3.loc[code, "營業收入-當月營收"])

        # 讀三季營收
        rev1 = float(data1.loc[code, "營業收入-上月比較增減(%)"])
        rev2 = float(data2.loc[code, "營業收入-上月比較增減(%)"])
        rev3 = float(data3.loc[code, "營業收入-上月比較增減(%)"])
        
        # 讀三季營業利益率
        op1 = float(data1.loc[code, "營業收入-去年同月增減(%)"])
        op2 = float(data2.loc[code, "營業收入-去年同月增減(%)"])
        op3 = float(data3.loc[code, "營業收入-去年同月增減(%)"])
        
        # 篩選三季營業利益率都 > 0
        if op1 > 0 and op2 > 0 and op3 > 0 and rev3 > 0 and rev2 > 0 and rev1 > 0:
            growth = r3 - r1  # 計算營收成長量
            growth_list.append((code, growth, r1, r2, r3, op1, op2, op3))

# 依照營收成長量排序，取前十
growth_list_sorted = sorted(growth_list, key=lambda x: x[1], reverse=True)[:10]

# 輸出結果
for item in growth_list_sorted:
    code, growth, r1, r2, r3, op1, op2, op3 = item
    print(f"{code} 營收: {r1}->{r2}->{r3} (成長 {growth})")

2408 營收: 15309988.0->15607184.0->18169760.0 (成長 2859772.0)
2344 營收: 11778411.0->11973254.0->14501399.0 (成長 2722988.0)
2027 營收: 9864568.0->9975843.0->11376755.0 (成長 1512187.0)
2337 營收: 3017005.0->3029953.0->4421909.0 (成長 1404904.0)
2548 營收: 1103070.0->1137637.0->1965711.0 (成長 862641.0)
2072 營收: 1176244.0->1336414.0->1435546.0 (成長 259302.0)
7610 營收: 268780.0->269795.0->463188.0 (成長 194408.0)
2465 營收: 466630.0->477648.0->624910.0 (成長 158280.0)
6534 營收: 219296.0->272876.0->377331.0 (成長 158035.0)
9958 營收: 1436160.0->1551164.0->1581026.0 (成長 144866.0)


In [12]:
growth_list = []

for code in data2.index:
    if code in data1.index and code in data3.index:

        r1 = float(data1.loc[code, "營業收入-當月營收"])
        r2 = float(data2.loc[code, "營業收入-當月營收"])
        r3 = float(data3.loc[code, "營業收入-當月營收"])

        yoy1 = float(data1.loc[code, "營業收入-去年同月增減(%)"])
        yoy2 = float(data2.loc[code, "營業收入-去年同月增減(%)"])
        yoy3 = float(data3.loc[code, "營業收入-去年同月增減(%)"])

        # ✅ 條件1：營收連續成長
        cond1 = (r1 < r2 < r3)

        # ✅ 條件2：年增率連三月 > 0
        cond2 = (yoy1 > 0 and yoy2 > 0 and yoy3 > 0)

        if cond1 and cond2:
            growth_rate = (r3 - r1) / r1
            growth_list.append((code, growth_rate, r1, r2, r3))

# 排序
top10 = sorted(growth_list, key=lambda x: x[1], reverse=True)[:10]

for code, g, r1, r2, r3 in top10:
    print(f"{code} {r1}→{r2}→{r3} 成長率: {g:.2%}")

6541 17649.0→22721.0→92457.0 成長率: 423.87%
3054 13427.0→57808.0→64646.0 成長率: 381.46%
7786 319988.0→341377.0→1111610.0 成長率: 247.39%
6477 41447.0→76996.0→106559.0 成長率: 157.10%
2425 627220.0→932779.0→1539516.0 成長率: 145.45%
6869 355674.0→562434.0→746320.0 成長率: 109.83%
8112 25631121.0→31218239.0→53315182.0 成長率: 108.01%
2548 1103070.0→1137637.0→1965711.0 成長率: 78.20%
7610 268780.0→269795.0→463188.0 成長率: 72.33%
6534 219296.0→272876.0→377331.0 成長率: 72.06%


In [ ]:
import yfinance as yf
import pandas as pd
from tqdm import tqdm
import random

# 固定 seed（很重要，避免結果亂跳）
random.seed(42)



# 台股常見範圍（可再擴）
#tickers = [f"{i}.TW" for i in range(1100, 9999)]
#tickers = random.sample(tickers, 300)

# 如果你要先測試，可以先縮小：
tickers = ["5469.TW", "6191.TW", "2912.TW", "2383.TW", "5388.TW", "5434.TW", "2542.TW", "2442.TW", 
           "2329.TW", "3044.TW", "2356.TW", "2303.TW", "2308.TW", "3037.TW", "5203.TW", "1513.TW",
           "3293.TW", "3515.TW", "3211.TW", "1580.TW", "6176.TW", "2439.TW"]


# =========================
# ② 單檔抓資料函數
# =========================
def get_data(ticker):
    try:
        df = yf.download(ticker, start="2020-01-01", progress=False)

        if df is None or df.empty:
            return None

        df = df.reset_index()

        # 保險：避免 MultiIndex
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # 標準化欄位
        df.columns = [str(c).lower() for c in df.columns]

        required = ["date", "open", "high", "low", "close", "volume"]
        if any(c not in df.columns for c in required):
            return None

        df["ticker"] = ticker

        return df[["date","open","high","low","close","volume","ticker"]]

    except:
        return None


# =========================
# ③ 批量下載
# =========================
data_list = []

for t in tqdm(tickers):
    d = get_data(t)
    if d is not None:
        data_list.append(d)

# =========================
# ④ 合併
# =========================
df = pd.concat(data_list, ignore_index=True)

df = df.sort_values(["ticker", "date"]).reset_index(drop=True)


 76%|███████▌  | 16/21 [00:06<00:02,  2.46it/s]$3293.TW: possibly delisted; no timezone found

1 Failed download:
['3293.TW']: possibly delisted; no timezone found
 86%|████████▌ | 18/21 [00:06<00:00,  3.35it/s]$3211.TW: possibly delisted; no timezone found

1 Failed download:
['3211.TW']: possibly delisted; no timezone found
$1580.TW: possibly delisted; no timezone found

1 Failed download:
['1580.TW']: possibly delisted; no timezone found
100%|██████████| 21/21 [00:07<00:00,  2.87it/s]


In [ ]:
import numpy as np
from hurst import compute_Hc

def last_n_hurst(series, window=60, n=5):
    series = series.dropna()
    
    
    
    hurst_list = []
    
    for i in range(n, 0, -1):
        window_data = series.iloc[-window-i+1:-i+1] if i > 1 else series.iloc[-window:]
        log_ret = np.log(window_data).diff().dropna()
        
        try:
            if len(log_ret) < 30:
                hurst_list.append(np.nan)
                pass
            else:
                H, _, _ = compute_Hc(log_ret.values, kind="change")
                hurst_list.append(H)
        except:
            hurst_list.append(np.nan)
    
    return pd.Series(hurst_list)

hurst_5d = (
    df.sort_values(["ticker", "date"])
      .groupby("ticker")["close"]
      .apply(lambda x: last_n_hurst(x, window=120, n=5).values)
      .apply(pd.Series)
)

hurst_5d.columns = ["H_t-4", "H_t-3", "H_t-2", "H_t-1", "H_t"]
hurst_5d

,H_t-4,H_t-3,H_t-2,H_t-1,H_t
ticker,,,,,
1513.TW,0.389761,0.391818,0.389278,0.408588,0.449071
2303.TW,0.592657,0.592517,0.575791,0.553515,0.579026
2308.TW,0.636036,0.652943,0.656858,0.644241,0.620744
2329.TW,0.425469,0.394693,0.397869,0.406407,0.431662
2356.TW,0.318301,0.350999,0.379674,0.385611,0.404697
2383.TW,0.811025,0.827440,0.816406,0.791378,0.785784
2442.TW,0.565363,0.543340,0.587996,0.596565,0.568494
2542.TW,0.492994,0.474589,0.471798,0.441328,0.489241
2912.TW,0.585547,0.568206,0.556727,0.518294,0.532338
